# ADME public data splitting

---

This notebook performs the scaffold-based train/calibration/validation split of the preprocessed ADME public dataset (produced by `Prepare_data.ipynb`).

**Note:** This notebook relies on `chemprop`, so it must be run using the `chemprop-env` conda environment.

In [ ]:
import pandas as pd
from rdkit import Chem
from chemprop.data.splitting import make_split_indices

In [ ]:
# Load filtered data
inputFile = "data/ADME_public_set_3521_preprocessed20260908.csv"
df_filtered = pd.read_csv(inputFile)

In [ ]:
# Generate RDKit molecule objects
df_filtered['mol'] = df_filtered['Structure'].apply(lambda x: Chem.MolFromSmiles(x))

In [ ]:
# Define the endpoints dictionary
endpoints_dict = {'CLint': ['rLM LogCLint', 'hLM LogCLint'],
                  'MDR1': ['MDCK-MDR1_LogER'],
                  'PPB': ['LogFu-Rat', 'LogFu-Human']}

for endpoint, models_list in endpoints_dict.items():
    print(f'\n### {endpoint} ###\n')

    # Select endpoint data
    df_endpoint = df_filtered[['Id', 'Structure', 'mol'] + models_list]
    df_endpoint = df_endpoint.dropna(subset=models_list, how='all').reset_index(drop=True)
    print(df_endpoint.shape)

    # Split data into training, calibration and test sets (scaffold-based)
    train_idxs, cal_idxs, val_idxs = make_split_indices(df_endpoint['mol'], split='SCAFFOLD_BALANCED', sizes=(0.5, 0.3, 0.2))
    train_idxs, cal_idxs, val_idxs = list(train_idxs[0]), list(cal_idxs[0]), list(val_idxs[0])
    df_endpoint['Subset'] = ''
    df_endpoint.loc[train_idxs, 'Subset'] = 'Training'
    df_endpoint.loc[cal_idxs, 'Subset'] = 'Calibration'
    df_endpoint.loc[val_idxs, 'Subset'] = 'Validation'

    # Select training data and further split it into training and validation
    # sets for model training and early stopping (scaffold-based)
    training_data = df_endpoint.loc[df_endpoint['Subset'] == 'Training'].reset_index(drop=True)
    test_data = df_endpoint.loc[df_endpoint['Subset'] != 'Training'].reset_index(drop=True)
    train_idxs, _, val_idxs = make_split_indices(training_data['mol'], split='SCAFFOLD_BALANCED', sizes=(0.9, 0, 0.1))
    train_idxs, val_idxs = list(train_idxs[0]), list(val_idxs[0])
    training_data['split'] = ''
    training_data.loc[train_idxs, 'split'] = 'train'
    training_data.loc[val_idxs, 'split'] = 'val'
    test_data['split'] = 'test'
    df_endpoint = pd.concat([training_data, test_data], ignore_index=True)
    summary_time_split = pd.DataFrame([df_endpoint[f'Subset'].value_counts()[['Training','Calibration','Validation']], (df_endpoint[f'Subset'].value_counts()/df_endpoint.shape[0]*100)[['Training','Calibration','Validation']]], index=['# Cpds','% Cpds'])
    print(summary_time_split.round())
    summary_time_split = pd.DataFrame([df_endpoint[f'split'].value_counts()[['train','val','test']], (df_endpoint[f'split'].value_counts()/df_endpoint.shape[0]*100)[['train','val','test']]], index=['# Cpds','% Cpds'])
    print(summary_time_split.round())

    # Save preprocessed endpoint dataset with data subsets and splits
    df_endpoint.to_csv(f'./data/{endpoint}_dataset_with_splits.csv')